# M4c — multilabel ATT&CK con grafo (4 experimentos)

Pregunta: M4 (LogReg texto+tabular) y M4b (XGB texto+tabular) no separan T1041/T1567. ¿El grafo ayuda?

- E0 tabular XGB (sin texto ni grafo) — baseline
- E1 texto+tabular XGB (= M4b, reproducido)
- E2 tabular+grafo XGB (NUEVO, sin texto)
- E3 texto+tabular+grafo XGB (FULL M4c, NUEVO)

Mismo protocolo: threat live (suspicious+malicious), 6 técnicas [T1190,T1041,T1567,T1204,T1486,T1566], split temporal 80/20, XGB binario por técnica con scale_pos_weight. Métricas: micro/macro-F1, hamming, subset-acc + P/R/F1 por técnica.

In [1]:
import json, time
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
ROOT = ROOT if (ROOT/'pyproject.toml').exists() else ROOT.parent
print('ROOT:', ROOT)
CATS = ['message_type','stream_name','product_name','vendor_name','pipeline','org_id','action','severity','protocol']
GFEATS = ['n_neigh','n_hosts','n_users','frac_block','frac_fw','frac_authfail','n_types','span_min']
TECHS = ['T1190','T1041','T1567','T1204','T1486','T1566']
W = 300
COLS = ['label_binary','timestamp','event_time','message_sanitized','attack_techniques','message_type','stream_name','product_name','vendor_name','pipeline','org_id','action','severity','protocol','src_ip','dst_ip','username']
df = pd.read_parquet(ROOT/'data/signals/signals.parquet', columns=COLS)
th = df[df.label_binary!='benign'].copy().reset_index(drop=True)
print('threat:', th.shape, th.label_binary.value_counts().to_dict())
def parse(v):
    try:
        t = json.loads(v) if v else []
        return [x for x in t if x in TECHS]
    except: return []
Y = np.zeros((len(th), len(TECHS)), dtype=int)
for i,v in enumerate(th.attack_techniques.fillna('')):
    for t in parse(v): Y[i, TECHS.index(t)] = 1
print('positivos por técnica:', {t: int(Y[:,j].sum()) for j,t in enumerate(TECHS)})

ROOT: C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto


threat: (114521, 17) {'suspicious': 106793, 'malicious': 7728}


positivos por técnica: {'T1190': 114507, 'T1041': 7742, 'T1567': 7728, 'T1204': 20, 'T1486': 0, 'T1566': 0}


In [2]:
# features base P1-P3 + split temporal
th['ts'] = pd.to_datetime(th['timestamp'], unit='s', utc=True)
th['hour'] = th.ts.dt.hour
th['msg_len'] = th.message_sanitized.fillna('').str.len()
th['lag_h'] = ((th['timestamp']-th['event_time'])/3600).clip(-1e4,1e4).fillna(-1)
for c in CATS: th[c] = th[c].fillna('').astype(str)
msg = th.message_sanitized.fillna('')
cutoff = th['timestamp'].quantile(0.80)
tr = (th['timestamp'] <= cutoff).to_numpy(); te = ~tr
print(f'train/test: {tr.sum()}/{te.sum()} corte: {pd.to_datetime(cutoff, unit="s", utc=True)}')

train/test: 91617/22904 corte: 2024-07-27 05:13:54.964786053+00:00


In [3]:
# grafo: índice full-data 2M (como M3) + agregación solo threat (114k). Celda lenta (~5-10 min), una sola vez.
fcols = ['timestamp','org_id','message_type','action','src_ip','dst_ip','username']
full = pd.read_parquet(ROOT/'data/signals/signals.parquet', columns=fcols)
for c in ['src_ip','dst_ip','username','org_id','message_type','action']:
    full[c] = full[c].fillna('').astype(str)
ts = full['timestamp'].to_numpy()
src = full['src_ip'].to_numpy(); dst = full['dst_ip'].to_numpy(); usr = full['username'].to_numpy()
org = full['org_id'].to_numpy(); mmsg = full['message_type'].to_numpy(); act = full['action'].to_numpy()
# mapa artifact? threat son filas de signals: necesitamos posiciones de threat en full. full está en orden parquet = mismo orden que df.
threat_idx = df[df.label_binary!='benign'].index.to_numpy()
print('threat_idx:', len(threat_idx))
index, ent = {}, {}
pos = pd.Series(np.arange(len(full)))
for c, vals in (('s',src),('d',dst),('u',usr)):
    key = np.char.add(org, np.char.add('|', vals.astype(str)))
    if c=='s': ents=key
    for k,s in pos.groupby(key):
        if k.endswith('|'): continue
        arr = s.to_numpy(); index[(c,k)] = arr[np.argsort(ts[arr])]
print('entidades:', len(index))
Es = np.char.add(org[threat_idx], np.char.add('|', src[threat_idx].astype(str)))
Ed = np.char.add(org[threat_idx], np.char.add('|', dst[threat_idx].astype(str)))
Eu = np.char.add(org[threat_idx], np.char.add('|', usr[threat_idx].astype(str)))
out = np.zeros((len(threat_idx),8), dtype=np.float32)
t0=time.time()
for j,i in enumerate(threat_idx):
    cand=set()
    for c,k in (('s',Es[j]),('d',Ed[j]),('u',Eu[j])):
        if k.endswith('|'): continue
        arr=index.get((c,k))
        if arr is None: continue
        lo=np.searchsorted(ts[arr], ts[i]-W); hi=np.searchsorted(ts[arr], ts[i]+W)
        cand.update(arr[lo:hi].tolist())
    cand.discard(i)
    if cand:
        nb=np.fromiter(cand,dtype=np.int64)
        hosts=set(src[nb].tolist())|set(dst[nb].tolist()); hosts.discard('')
        users=set(usr[nb].tolist()); users.discard('')
        a,m,t=mmsg[nb],act[nb],ts[nb]
        # nota: en full mmsg=message_type, act=action (nombres cruzados arriba, se corrige aquí)
        out[j]=[len(nb),len(hosts),len(users),float(np.mean(act[nb]=='block')),float(np.mean(mmsg[nb]=='firewall_action')),float(np.mean(mmsg[nb]=='auth_failure')),float(len(set(mmsg[nb].tolist()))),float(ts[nb].max()-ts[nb].min())/60.0]
    if (j+1)%20000==0: print(f'{j+1}/{len(threat_idx)} {(time.time()-t0)/60:.1f}min', flush=True)
G = pd.DataFrame(out, columns=GFEATS)
print('G.describe:'); print(G.describe())
print(f'tiempo total {(time.time()-t0)/60:.1f} min')

threat_idx: 114521


entidades: 14773


20000/114521 0.3min


40000/114521 0.4min


60000/114521 0.5min


80000/114521 0.6min


100000/114521 0.7min


G.describe:
             n_neigh        n_hosts        n_users     frac_block  \
count  114521.000000  114521.000000  114521.000000  114521.000000   
mean      851.863770      14.236691       0.010295       0.996665   
std      1735.640869      11.161150       0.382407       0.055968   
min         0.000000       0.000000       0.000000       0.000000   
25%        95.000000       3.000000       0.000000       1.000000   
50%       234.000000      10.000000       0.000000       1.000000   
75%       506.000000      22.000000       0.000000       1.000000   
max      7161.000000      47.000000      47.000000       1.000000   

             frac_fw  frac_authfail        n_types       span_min  
count  114521.000000  114521.000000  114521.000000  114521.000000  
mean        0.828698       0.000367       1.006016       8.480283  
std         0.376375       0.013714       0.116071       2.461203  
min         0.000000       0.000000       0.000000       0.000000  
25%         1.000000      

In [4]:
# matrices: TAB (one-hot+nums), TXT (tfidf 5k), G (8 feats)
from scipy.sparse import hstack, csr_matrix
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
tab = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', max_categories=30), CATS), ('num','passthrough',['hour','msg_len','lag_h'])])
Atr = tab.fit_transform(th[tr][CATS+['hour','msg_len','lag_h']]); Ate = tab.transform(th[te][CATS+['hour','msg_len','lag_h']])
vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True, min_df=5)
Ttr = vec.fit_transform(msg[tr].tolist()); Tte = vec.transform(msg[te].tolist())
Gtr = csr_matrix(G.to_numpy()[tr]); Gte = csr_matrix(G.to_numpy()[te])
EXP = {'E0_tab': (Atr,Ate), 'E1_txt+tab': (hstack([Ttr,Atr]).tocsr(), hstack([Tte,Ate]).tocsr()), 'E2_tab+grafo': (hstack([Atr,Gtr]).tocsr(), hstack([Ate,Gte]).tocsr()), 'E3_txt+tab+grafo': (hstack([Ttr,Atr,Gtr]).tocsr(), hstack([Tte,Ate,Gte]).tocsr())}
for k,(a,b) in EXP.items(): print(k, a.shape, b.shape)

E0_tab (91617, 54) (22904, 54)
E1_txt+tab (91617, 5054) (22904, 5054)
E2_tab+grafo (91617, 62) (22904, 62)
E3_txt+tab+grafo (91617, 5062) (22904, 5062)


In [5]:
# 4 experimentos × 6 XGB binarios. Celda lenta (~minutos), una sola vez.
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, hamming_loss, accuracy_score, classification_report
res={}
for name,(Xtr,Xte) in EXP.items():
    t0=time.time(); P=np.zeros_like(Y[te]); skipped=[]
    for j,t in enumerate(TECHS):
        pos=int(Y[tr][:,j].sum()); neg=int(tr.sum())-pos
        if pos<2: skipped.append(t); continue
        clf=XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.08,subsample=0.8,colsample_bytree=0.8,tree_method='hist',objective='binary:logistic',scale_pos_weight=neg/pos,eval_metric='logloss',n_jobs=-1,random_state=42)
        clf.fit(Xtr, Y[tr][:,j])
        P[:,j]=(clf.predict_proba(Xte)[:,1]>=0.5).astype(int)
    micro=round(float(f1_score(Y[te],P,average='micro')),4); macro=round(float(f1_score(Y[te],P,average='macro')),4)
    ham=round(float(hamming_loss(Y[te],P)),4); sub=round(float(accuracy_score(Y[te],P)),4)
    res[name]={'micro_f1':micro,'macro_f1':macro,'hamming':ham,'subset_acc':sub,'skipped':skipped,'secs':int(time.time()-t0)}
    print(f'=== {name} micro={micro} macro={macro} ham={ham} sub={sub} ({res[name]["secs"]}s) ===')
    print(classification_report(Y[te],P,target_names=TECHS,zero_division=0))
print(json.dumps(res,indent=2))
Path(ROOT/'models'/'m4c_exps.json').write_text(json.dumps(res,indent=2))
print('guardado models/m4c_exps.json')

C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


=== E0_tab micro=0.8536 macro=0.2643 ham=0.0577 sub=0.8269 (3s) ===
              precision    recall  f1-score   support

       T1190       1.00      1.00      1.00     22892
       T1041       0.26      0.03      0.06      3733
       T1567       0.24      0.03      0.05      3721
       T1204       0.31      1.00      0.48        10
       T1486       0.00      0.00      0.00         0
       T1566       0.00      0.00      0.00         0

   micro avg       0.97      0.76      0.85     30356
   macro avg       0.30      0.34      0.26     30356
weighted avg       0.82      0.76      0.77     30356
 samples avg       0.99      0.89      0.91     30356



C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


=== E1_txt+tab micro=0.7926 macro=0.2557 ham=0.089 sub=0.7334 (205s) ===
              precision    recall  f1-score   support

       T1190       1.00      1.00      1.00     22892
       T1041       0.08      0.06      0.07      3733
       T1567       0.08      0.06      0.07      3721
       T1204       0.24      1.00      0.39        10
       T1486       0.00      0.00      0.00         0
       T1566       0.00      0.00      0.00         0

   micro avg       0.82      0.77      0.79     30356
   macro avg       0.23      0.35      0.26     30356
weighted avg       0.77      0.77      0.77     30356
 samples avg       0.92      0.90      0.87     30356



C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


=== E2_tab+grafo micro=0.8571 macro=0.2957 ham=0.0562 sub=0.831 (4s) ===
              precision    recall  f1-score   support

       T1190       1.00      1.00      1.00     22892
       T1041       0.35      0.04      0.07      3733
       T1567       0.33      0.03      0.06      3721
       T1204       0.48      1.00      0.65        10
       T1486       0.00      0.00      0.00         0
       T1566       0.00      0.00      0.00         0

   micro avg       0.98      0.76      0.86     30356
   macro avg       0.36      0.35      0.30     30356
weighted avg       0.84      0.76      0.77     30356
 samples avg       0.99      0.90      0.92     30356



=== E3_txt+tab+grafo micro=0.7907 macro=0.2779 ham=0.09 sub=0.7293 (169s) ===
              precision    recall  f1-score   support

       T1190       1.00      1.00      1.00     22892
       T1041       0.08      0.06      0.07      3733
       T1567       0.08      0.06      0.07      3721
       T1204       0.36      1.00      0.53        10
       T1486       0.00      0.00      0.00         0
       T1566       0.00      0.00      0.00         0

   micro avg       0.81      0.77      0.79     30356
   macro avg       0.25      0.35      0.28     30356
weighted avg       0.77      0.77      0.77     30356
 samples avg       0.92      0.90      0.86     30356

{
  "E0_tab": {
    "micro_f1": 0.8536,
    "macro_f1": 0.2643,
    "hamming": 0.0577,
    "subset_acc": 0.8269,
    "skipped": [
      "T1486",
      "T1566"
    ],
    "secs": 3
  },
  "E1_txt+tab": {
    "micro_f1": 0.7926,
    "macro_f1": 0.2557,
    "hamming": 0.089,
    "subset_acc": 0.7334,
    "skipped": [
      "T1

C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## Lectura (completa cuando corra)
- Si E2/E3 no suben macro (>0.30), veredicto: ni con grafo se separan T1041/T1567 a nivel evento → son priors del tier, van como hint (oráculo M4), no como modelo. Cierra ablación ATT&CK igual que M1→M2→M3 en tiers.
- Si E2/E3 sí suben T1204 o cola, repórtalo como único aporte del grafo en multilabel.